## ✅ Step 1: Import Required Libraries

In [1]:
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import torch

/home/shuaib/anaconda3/envs/fake-news-transformers/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## ✅ Step 2: Load Your Dataset

Since i don't have my own data, let's use a ready-made dataset from scikit-learn: the fetch_20newsgroups dataset. I'll simplify it into a binary fake news detection task using two categories:

"sci.space" (real news)

"talk.politics.misc" (fake news, just for this demo!)

This is NOT an actual fake news dataset, but it helps us simulate the task.

### Step 3: Load two categories from 20 Newsgroups

In [2]:
categories = ['sci.space', 'talk.politics.misc']  # simulating real vs fake

In [3]:
data = fetch_20newsgroups(subset="all", categories=categories, remove=("headers", "footers", "qoutes"))
data.keys()

dict_keys(['data', 'filenames', 'target_names', 'target', 'DESCR'])

In [4]:
X = data.data
y = data.target
y

array([0, 0, 0, ..., 0, 0, 1])

In [5]:
labels = [0 if target == data.target_names.index('sci.space') else 1 for target in data.target]

### Step 4: Train/test split

In [6]:
train_texts, test_texts, train_labels, test_labels = train_test_split(X, labels, test_size=0.25, random_state=42, stratify=labels)

### Step 5: Load tokenizer and tokenize

In [7]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
train_encodings = tokenizer(train_texts, truncation = True, padding = True, max_length = 512)
test_encodings = tokenizer(test_texts, truncation = True, padding = True, max_length = 512)


In [8]:
train_encodings 

{'input_ids': [[101, 1999, 3720, 1026, 6109, 10790, 2581, 1012, 14748, 22394, 2683, 23823, 4859, 2869, 2290, 1030, 24209, 19797, 2078, 1012, 8603, 2226, 1012, 6187, 1028, 3897, 5280, 1026, 7842, 8630, 2869, 2290, 1030, 24209, 19797, 2078, 1012, 8603, 2226, 1012, 6187, 1028, 7009, 1024, 1028, 2023, 2003, 3810, 2046, 1005, 2054, 1005, 1055, 1037, 4231, 15058, 2204, 2005, 1005, 1010, 1998, 1045, 11276, 1028, 2025, 2000, 2695, 2043, 1045, 1005, 2310, 1037, 3634, 2070, 5976, 8466, 2000, 2175, 1010, 2021, 1045, 2052, 1028, 2228, 2008, 1996, 2613, 3114, 2000, 2031, 1037, 4231, 2918, 2003, 3171, 1012, 1028, 1028, 2144, 2619, 2007, 2686, 3068, 2097, 3653, 23545, 8231, 2031, 1037, 2172, 1028, 3469, 1043, 16275, 2084, 2027, 2052, 1035, 2302, 1035, 2686, 3068, 1010, 2776, 1010, 1028, 2027, 2097, 3432, 2022, 2583, 2000, 8984, 2062, 4933, 1012, 2065, 1045, 3191, 2017, 2157, 1010, 2017, 1005, 2128, 3038, 1999, 11305, 2008, 1010, 2007, 1037, 3469, 4610, 1010, 3741, 2097, 2031, 2062, 19258, 5649, 5029,

### Step 6: Convert to PyTorch dataset

In [9]:
class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["label"] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return(self.lables)


In [10]:
train_dataset = NewsDataset(train_encodings, train_labels)
test_dataset = NewsDataset(test_encodings, test_labels)

### Step 7: Load model

In [11]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels = 2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Step 8: Training arguments

In [12]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=100,
    # evaluation_strategy="epoch",
    logging_dir='./logs',
)

### Step 9: Train the model

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

### Step 10: Evaluate the model

In [ ]:
preds_output = trainer.predict(test_dataset)
preds = torch.argmax(torch.tensor(preds_output.predictions), axis = 1)
print("Accuracy: ", accuracy_score(test_labels, preds))
print(classification_report(test_lables, preds, target_names = ["Real", "Fake"]))

In [ ]:
# Step 1: Install dependencies
# !pip install transformers scikit-learn torch

# Step 2: Import libraries
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import torch

# Step 3: Load two categories from 20 Newsgroups
categories = ['sci.space', 'talk.politics.misc']  # simulating real vs fake

data = fetch_20newsgroups(subset='all', categories=categories, remove=('headers', 'footers', 'quotes'))

# Create labels: 0 = real, 1 = fake
texts = data.data
labels = [0 if target == data.target_names.index('sci.space') else 1 for target in data.target]

# Step 4: Train/test split
train_texts, test_texts, train_labels, test_labels = train_test_split(texts, labels, test_size=0.2, random_state=42)

# Step 5: Load tokenizer and tokenize
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=512)

# Step 6: Convert to PyTorch dataset
class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(train_encodings, train_labels)
test_dataset = NewsDataset(test_encodings, test_labels)

# Step 7: Load model
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# Step 8: Training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=100,
    # evaluation_strategy="epoch",
    logging_dir='./logs',
)

# Step 9: Train the model
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

# Step 10: Evaluate the model
preds_output = trainer.predict(test_dataset)
preds = torch.argmax(torch.tensor(preds_output.predictions), axis=1)

print("Accuracy:", accuracy_score(test_labels, preds))
print(classification_report(test_labels, preds, target_names=["Real", "Fake"]))
